# Healthcare Claims – Daily Landing Data Generator

Creates three **synthetic healthcare datasets** and writes them to your Unity Catalog landing volume:

- `members`: approximately 5,000 rows
- `providers`: approximately 300 rows
- `claims`: 20,000 rows for Day 1

**Target volume:** `/Volumes/claim-etl/bronze/landing/healthcare/daily/`

The generator creates realistic-looking relationships: every claim references an existing member and provider, service dates are plausible, and paid amounts/statuses are consistent.

> All records are synthetic and must not be used as real patient or claims data.

In [0]:
# 1. Configuration
from datetime import date, datetime, timedelta
import random

CATALOG = "claim-etl"
SCHEMA = "bronze"
VOLUME = "landing"

DROP_DATE = date.today() - timedelta(days=1)  # Change to date.today() if preferred

MEMBER_COUNT = 5_000
PROVIDER_COUNT = 300
CLAIM_COUNT = 20_000

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/healthcare/daily"
DAILY_PATH = f"{BASE_PATH}/drop_date={DROP_DATE.isoformat()}"

# Fixed seed makes the same date reproducible.
random.seed(DROP_DATE.isoformat())

print("Drop date:", DROP_DATE)
print("Landing path:", DAILY_PATH)

In [0]:
# 2. Reference data
first_names = [
    "Aarav", "Vivaan", "Aditya", "Arjun", "Ishaan", "Rohan", "Rahul",
    "Ananya", "Diya", "Aadhya", "Saanvi", "Meera", "Kavya", "Neha",
    "Priya", "Ira", "Vikram", "Kiran", "Nisha", "Anil"
]
last_names = [
    "Sharma", "Patel", "Reddy", "Iyer", "Nair", "Gupta", "Kumar",
    "Rao", "Mehta", "Das", "Menon", "Singh", "Joshi", "Pillai"
]
locations = {
    "Tamil Nadu": ["Chennai", "Coimbatore", "Madurai", "Salem"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangaluru", "Hubballi"],
    "Maharashtra": ["Mumbai", "Pune", "Nagpur", "Nashik"],
    "Telangana": ["Hyderabad", "Warangal", "Nizamabad"],
    "Delhi": ["New Delhi", "Dwarka", "Rohini"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara"],
    "West Bengal": ["Kolkata", "Howrah", "Durgapur"]
}
states = list(locations.keys())

plan_types = ["Basic", "Standard", "Premium", "Family"]
specialties = [
    "General Medicine", "Cardiology", "Orthopedics", "Pediatrics",
    "Dermatology", "Neurology", "Oncology", "Endocrinology",
    "Gastroenterology", "Ophthalmology"
]
diagnosis_codes = [
    "E11.9", "I10", "J06.9", "M54.5", "E78.5",
    "K21.9", "J45.909", "R51.9", "N39.0", "M17.9"
]
procedure_codes = [
    "99213", "99214", "80053", "85025", "93000",
    "71046", "73721", "36415", "90471", "80061"
]

def random_person_name():
    return f"{random.choice(first_names)} {random.choice(last_names)}"

def random_timestamp():
    return datetime.combine(
        DROP_DATE,
        datetime.min.time()
    ) + timedelta(
        hours=random.randint(7, 20),
        minutes=random.choice([0, 10, 15, 20, 30, 45])
    )

print("Reference values ready.")

In [0]:
# 3. Generate members (5,000 rows)
members = []

for i in range(1, MEMBER_COUNT + 1):
    state = random.choice(states)
    dob_start = date(1940, 1, 1)
    dob_end = date(2024, 12, 31)
    dob = dob_start + timedelta(days=random.randint(0, (dob_end - dob_start).days))

    members.append({
        "member_id": f"MEM{i:06d}",
        "name": random_person_name(),
        "dob": dob.isoformat(),
        "gender": random.choice(["F", "M", "Other"]),
        "state": state,
        "city": random.choice(locations[state]),
        "plan_type": random.choice(plan_types),
        "updated_at": random_timestamp().isoformat(sep=" ", timespec="seconds")
    })

print(f"Generated members: {len(members):,}")
print(members[0])

In [0]:
# 4. Generate providers (300 rows)
providers = []

for i in range(1, PROVIDER_COUNT + 1):
    state = random.choice(states)

    providers.append({
        "provider_id": f"PRV{i:04d}",
        "name": f"{random.choice(['City', 'Metro', 'National', 'Sunrise', 'Green', 'Care'])} "
                f"{random.choice(['Hospital', 'Medical Center', 'Clinic', 'Health Services'])} "
                f"{i:03d}",
        "specialty": random.choice(specialties),
        "state": state
    })

print(f"Generated providers: {len(providers):,}")
print(providers[0])

In [0]:
# 5. Generate claims (20,000 rows for Day 1)
# Every claim references a member_id and provider_id generated above.
claims = []

member_ids = [m["member_id"] for m in members]
provider_ids = [p["provider_id"] for p in providers]

claim_statuses = ["PAID", "PARTIALLY_PAID", "DENIED", "PENDING"]

for i in range(1, CLAIM_COUNT + 1):
    billed_amount = round(random.uniform(500, 100000), 2)
    claim_status = random.choices(
        claim_statuses,
        weights=[60, 15, 10, 15],
        k=1
    )[0]

    if claim_status == "PAID":
        paid_amount = billed_amount
    elif claim_status == "PARTIALLY_PAID":
        paid_amount = round(billed_amount * random.uniform(0.30, 0.85), 2)
    else:
        paid_amount = 0.00

    # Claims generally relate to services in the recent past.
    service_date = DROP_DATE - timedelta(days=random.randint(0, 90))

    claims.append({
        "claim_id": f"CLM-{DROP_DATE:%Y%m%d}-{i:06d}",
        "member_id": random.choice(member_ids),
        "provider_id": random.choice(provider_ids),
        "service_date": service_date.isoformat(),
        "diagnosis_code": random.choice(diagnosis_codes),
        "procedure_code": random.choice(procedure_codes),
        "billed_amount": billed_amount,
        "paid_amount": paid_amount,
        "claim_status": claim_status,
        "updated_at": random_timestamp().isoformat(sep=" ", timespec="seconds")
    })

print(f"Generated claims: {len(claims):,}")
print(claims[0])

In [0]:
# 6. Create Spark DataFrames
members_df = spark.createDataFrame(members)
providers_df = spark.createDataFrame(providers)
claims_df = spark.createDataFrame(claims)

print("Members:", members_df.count())
print("Providers:", providers_df.count())
print("Claims:", claims_df.count())

display(members_df.limit(5))
display(providers_df.limit(5))
display(claims_df.limit(5))

In [0]:
# 7. Write daily files to the landing volume
# Each dataset is written to its own folder.
# Spark creates part files and a _SUCCESS marker in each folder.

members_path = f"{DAILY_PATH}/members"
providers_path = f"{DAILY_PATH}/providers"
claims_path = f"{DAILY_PATH}/claims"

(members_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(members_path))

(providers_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(providers_path))

(claims_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(claims_path))

print("All three datasets written successfully.")
print(members_path)
print(providers_path)
print(claims_path)

In [0]:
# 8. Verify landing files
for path in [members_path, providers_path, claims_path]:
    print(f"\\nFiles under: {path}")
    display(dbutils.fs.ls(path))

In [0]:
# 9. Basic source-data quality checks
assert members_df.count() == MEMBER_COUNT, "Unexpected member count"
assert providers_df.count() == PROVIDER_COUNT, "Unexpected provider count"
assert claims_df.count() == CLAIM_COUNT, "Unexpected claim count"

assert members_df.select("member_id").distinct().count() == MEMBER_COUNT, "Duplicate member IDs"
assert providers_df.select("provider_id").distinct().count() == PROVIDER_COUNT, "Duplicate provider IDs"
assert claims_df.select("claim_id").distinct().count() == CLAIM_COUNT, "Duplicate claim IDs"

invalid_member_refs = claims_df.join(
    members_df.select("member_id"), on="member_id", how="left_anti"
).count()

invalid_provider_refs = claims_df.join(
    providers_df.select("provider_id"), on="provider_id", how="left_anti"
).count()

invalid_amounts = claims_df.filter(
    (claims_df.paid_amount < 0) |
    (claims_df.billed_amount < 0) |
    (claims_df.paid_amount > claims_df.billed_amount)
).count()

assert invalid_member_refs == 0, f"Claims with unknown members: {invalid_member_refs}"
assert invalid_provider_refs == 0, f"Claims with unknown providers: {invalid_provider_refs}"
assert invalid_amounts == 0, f"Claims with invalid amounts: {invalid_amounts}"

print("All source-data quality checks passed.")

In [0]:
from datetime import date, datetime, timedelta
import random

DAY_2 = DROP_DATE + timedelta(days=1)
DAY_3 = DROP_DATE + timedelta(days=2)

BASE_PATH = "/Volumes/claim-etl/bronze/landing/healthcare/daily"

MEMBER_UPDATES = 100
MEMBER_INSERTS = 20

PROVIDER_UPDATES = 10
PROVIDER_INSERTS = 2

DAY_2_CLAIMS = 5000
DAY_3_CLAIMS = 5500

In [0]:
current_members = {
    r["member_id"]: dict(r) for r in members
}

current_providers = {
    r["provider_id"]: dict(r) for r in providers
}

next_member_id = MEMBER_COUNT + 1
next_provider_id = PROVIDER_COUNT + 1

In [0]:
def generate_member_changes(event_date, seed):

    global next_member_id

    rng = random.Random(seed)
    events = []

    # UPDATE existing members
    selected_ids = rng.sample(
        list(current_members.keys()),
        MEMBER_UPDATES
    )

    for member_id in selected_ids:

        member = dict(current_members[member_id])

        change = rng.choice(["plan", "location", "name"])

        if change == "plan":
            member["plan_type"] = rng.choice(
                [p for p in plan_types
                 if p != member["plan_type"]]
            )

        elif change == "location":
            state = rng.choice(
                [s for s in states if s != member["state"]]
            )
            member["state"] = state
            member["city"] = rng.choice(locations[state])

        else:
            member["name"] = (
                f"{rng.choice(first_names)} "
                f"{rng.choice(last_names)}"
            )

        member["updated_at"] = datetime.combine(
            event_date,
            datetime.min.time()
        ).replace(hour=rng.randint(8, 18)).isoformat(sep=" ")

        member["operation"] = "UPDATE"

        events.append(member)

        current_members[member_id] = {
            k: v for k, v in member.items()
            if k != "operation"
        }

    # INSERT new members
    for _ in range(MEMBER_INSERTS):

        state = rng.choice(states)

        member = {
            "member_id": f"MEM{next_member_id:06d}",
            "name": f"{rng.choice(first_names)} {rng.choice(last_names)}",
            "dob": (
                date(1940, 1, 1) +
                timedelta(days=rng.randint(0, 30000))
            ).isoformat(),
            "gender": rng.choice(["F", "M", "Other"]),
            "state": state,
            "city": rng.choice(locations[state]),
            "plan_type": rng.choice(plan_types),
            "updated_at": datetime.combine(
                event_date,
                datetime.min.time()
            ).replace(hour=10).isoformat(sep=" "),
            "operation": "INSERT"
        }

        next_member_id += 1
        events.append(member)

        current_members[member["member_id"]] = {
            k: v for k, v in member.items()
            if k != "operation"
        }

    return events

In [0]:
def generate_provider_changes(event_date, seed):

    global next_provider_id

    rng = random.Random(seed)
    events = []

    # UPDATE existing providers
    selected_ids = rng.sample(
        list(current_providers.keys()),
        PROVIDER_UPDATES
    )

    for provider_id in selected_ids:

        provider = dict(current_providers[provider_id])

        if rng.choice(["specialty", "state"]) == "specialty":
            provider["specialty"] = rng.choice(
                [s for s in specialties
                 if s != provider["specialty"]]
            )
        else:
            provider["state"] = rng.choice(
                [s for s in states
                 if s != provider["state"]]
            )

        provider["operation"] = "UPDATE"

        events.append(provider)

        current_providers[provider_id] = {
            k: v for k, v in provider.items()
            if k != "operation"
        }

    # INSERT new providers
    for _ in range(PROVIDER_INSERTS):

        provider = {
            "provider_id": f"PRV{next_provider_id:04d}",
            "name": f"City Medical Center {next_provider_id}",
            "specialty": rng.choice(specialties),
            "state": rng.choice(states),
            "operation": "INSERT"
        }

        next_provider_id += 1
        events.append(provider)

        current_providers[provider["provider_id"]] = {
            k: v for k, v in provider.items()
            if k != "operation"
        }

    return events

In [0]:
def generate_daily_claims(event_date, count, seed):

    rng = random.Random(seed)
    events = []

    member_ids = list(current_members.keys())
    provider_ids = list(current_providers.keys())

    statuses = [
        "PAID",
        "PARTIALLY_PAID",
        "DENIED",
        "PENDING"
    ]

    for i in range(1, count + 1):

        billed = round(rng.uniform(500, 100000), 2)

        status = rng.choices(
            statuses,
            weights=[60, 15, 10, 15]
        )[0]

        if status == "PAID":
            paid = billed

        elif status == "PARTIALLY_PAID":
            paid = round(billed * rng.uniform(0.3, 0.85), 2)

        else:
            paid = 0.0

        service_date = (
            event_date -
            timedelta(days=rng.randint(0, 90))
        )

        events.append({
            "claim_id": f"CLM-{event_date:%Y%m%d}-{i:06d}",
            "member_id": rng.choice(member_ids),
            "provider_id": rng.choice(provider_ids),
            "service_date": service_date.isoformat(),
            "diagnosis_code": rng.choice(diagnosis_codes),
            "procedure_code": rng.choice(procedure_codes),
            "billed_amount": billed,
            "paid_amount": paid,
            "claim_status": status,
            "updated_at": datetime.combine(
                event_date,
                datetime.min.time()
            ).replace(hour=rng.randint(8, 18)).isoformat(sep=" ")
        })

    return events

In [0]:
# DAY 2
day2_members = generate_member_changes(DAY_2, 200)
day2_providers = generate_provider_changes(DAY_2, 201)
day2_claims = generate_daily_claims(
    DAY_2, DAY_2_CLAIMS, 202
)

# DAY 3
day3_members = generate_member_changes(DAY_3, 300)
day3_providers = generate_provider_changes(DAY_3, 301)
day3_claims = generate_daily_claims(
    DAY_3, DAY_3_CLAIMS, 302
)

print("Day 2:", len(day2_members), len(day2_providers), len(day2_claims))
print("Day 3:", len(day3_members), len(day3_providers), len(day3_claims))

In [0]:
def write_daily_drop(event_date, members_data,
                     providers_data, claims_data):

    root = f"{BASE_PATH}/drop_date={event_date}"

    datasets = {
        "members": members_data,
        "providers": providers_data,
        "claims": claims_data
    }

    for name, records in datasets.items():

        df = spark.createDataFrame(records)

        path = f"{root}/{name}"

        (df.coalesce(1)
           .write
           .mode("overwrite")
           .option("header", True)
           .csv(path))

        print(f"{name}: {df.count()} records -> {path}")


write_daily_drop(
    DAY_2,
    day2_members,
    day2_providers,
    day2_claims
)

write_daily_drop(
    DAY_3,
    day3_members,
    day3_providers,
    day3_claims
)

In [0]:
from collections import Counter

for day, member_events, provider_events in [
    ("Day 2", day2_members, day2_providers),
    ("Day 3", day3_members, day3_providers)
]:

    print(f"\n{day}")

    print("Member operations:",
          Counter(r["operation"] for r in member_events))

    print("Provider operations:",
          Counter(r["operation"] for r in provider_events))

## Landing layout

```text
/Volumes/claim-etl/bronze/landing/healthcare/daily/
└── drop_date=YYYY-MM-DD/
    ├── members/
    │   └── part-....csv
    ├── providers/
    │   └── part-....csv
    └── claims/
        └── part-....csv
```

## Notes for the next pipeline stage

- **Landing:** raw CSV files as delivered by the simulated source.
- **Bronze:** read the CSVs, enforce explicit schemas, add ingestion metadata, and persist as Delta tables.
- **Silver:** validate member/provider references, standardize codes, deduplicate claims, and apply business rules.
- **Gold:** build reporting tables such as claims by provider, paid amount by plan, and denial rates.

This is a Day 1 full dataset. For later daily runs, decide whether members/providers should be full snapshots or only changed records, and whether claims should contain only newly arrived claims. The current notebook regenerates all three datasets for the selected drop date.